# Wrangle marker genes for HLCA 

In [2]:
import yaml
import pandas as pd

In [33]:
df = pd.read_excel(
    'data/input/HLCA_supplement.xlsx',
    sheet_name='6 - marker genes',
    header=1,
    index_col=0,
)

In [34]:
# 1. Create the MultiIndex from column names
# Splitting only on the FIRST underscore to preserve cell names with spaces
col_split = df.columns.str.extract(r'^([^_]+)_(.*)')
df.columns = pd.MultiIndex.from_arrays(
    [col_split[0], col_split[1]], 
    names=['cell_type', 'attribute']
)

In [50]:
# 2. Use the new future_stack implementation
# This silences the warning and prepares your code for Pandas 3.0
df_reshaped = df.stack(
    level='cell_type',
    future_stack=True
).reset_index().drop(
    columns='level_0'
).sort_values(
    ['cell_type', 'marker_for', 'marker_reference']
).dropna()
df_reshaped.columns.name = None
df_reshaped

,cell_type,marker,marker_for,marker_reference
366,AT0,SFTPB,AT0,Airway epithelium
427,AT0,SCGB3A2,AT0,Airway epithelium
488,AT0,SFTA2,AT0,Airway epithelium
183,AT0,IGFBP2,Airway epithelium,Epithelial
244,AT0,SERPINF1,Airway epithelium,Epithelial
...,...,...,...,...
121,pre-TB secretory,EPCAM,Epithelial,Full atlas
182,pre-TB secretory,ELF3,Epithelial,Full atlas
426,pre-TB secretory,SFTPB,"pre-TB secretory (poss. lowly expressed, non-u...",Airway epithelium
487,pre-TB secretory,RNASE1,"pre-TB secretory (poss. lowly expressed, non-u...",Airway epithelium


In [61]:
specific_markers = df_reshaped.query(
    'cell_type == marker_for'
)

In [65]:
specific_markers

,cell_type,marker,marker_for,marker_reference
366,AT0,SFTPB,AT0,Airway epithelium
427,AT0,SCGB3A2,AT0,Airway epithelium
488,AT0,SFTA2,AT0,Airway epithelium
367,AT1,CLIC5,AT1,Alveolar epithelium
428,AT1,SPOCK2,AT1,Alveolar epithelium
...,...,...,...,...
544,Subpleural fibroblasts,MMP23B,Subpleural fibroblasts,Fibroblast lineage
363,T cells proliferating,CENPW,T cells proliferating,Lymphoid
424,T cells proliferating,TK1,T cells proliferating,Lymphoid
485,T cells proliferating,MKI67,T cells proliferating,Lymphoid


In [69]:
marker_dict = df_reshaped.groupby('cell_type')['marker'].apply(list).to_dict()

In [70]:
with open('configs/HLCA/gene_sets_full.yaml', 'w') as f:
    yaml.safe_dump(marker_dict, f, sort_keys=False, default_flow_style=False)